# Week 96, Day 4: Reviewing pull requests

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w096d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('pr.diff', 'w', encoding='utf-8') as f:
    f.write('diff --git a/Lib/wraptext.py b/Lib/wraptext.py\n--- a/Lib/wraptext.py\n+++ b/Lib/wraptext.py\n@@ -24,3 +24,16 @@ def shorten(text, width, placeholder=" [...]"):\n     if not kept:\n         raise ValueError("placeholder too large for max width")\n     return " ".join(kept) + placeholder\n+\n+\n+def shorten_all(texts, width, seen=[]):\n+    results = []\n+    for text in texts:\n+        try:\n+            results.append(shorten(text, width))\n+        except:\n+            results.append(None)\n+        if text == None:\n+            print("skipped", text)\n+        seen.append(text)\n+    return results\ndiff --git a/Lib/test/test_wraptext.py b/Lib/test/test_wraptext.py\n--- a/Lib/test/test_wraptext.py\n+++ b/Lib/test/test_wraptext.py\n@@ -30,2 +30,5 @@ class TestShorten(unittest.TestCase):\n     def test_width_too_small(self):\n         with self.assertRaises(ValueError):\n+            shorten("one two", 3)\n+\n+    def test_all(self):\n+        self.assertEqual(shorten_all(["a b"], 10), ["a b"])\n')

**Predict before you run:** what will this print?



In [ ]:
def shorten_all(texts, seen=[]):
    seen.extend(texts)
    return len(seen)

print(shorten_all(["a"]), shorten_all(["b"]), shorten_all(["c"]))

> **Key idea.** Review the issue, the description, then the diff: should it happen, is it correct (run it; watch for bare `except:`, `== None`, mutable defaults, prints, `eval`), is it tested, is it complete (docs, NEWS), is it clear. Label comments (issue, suggestion, nitpick, question, praise) and mark blocking or non-blocking. Describe code, not people; suggest; praise sincerely. Sum up: approve, comment, or request changes.

### Exercise: Comments that say what they are

Write `conventional(label, subject, decoration=None, where=None)`: return `"LABEL (DECORATION): SUBJECT"` (or `"LABEL: SUBJECT"` without a decoration), with `"WHERE: "` in front if `where` is given. Raise `ValueError("unknown label: L")` unless the label is one of `praise`, `nitpick`, `suggestion`, `issue`, `question`, `thought`, and `ValueError("unknown decoration: D")` unless a given decoration is `blocking` or `non-blocking`.

In [ ]:
def conventional(label, subject, decoration=None, where=None):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_comments():\n    \"the label first\"\n    assert conventional(\"issue\", \"a bare except: hides KeyboardInterrupt\", \"blocking\", \"Lib/wraptext.py:34\") == (\n        \"Lib/wraptext.py:34: issue (blocking): a bare except: hides KeyboardInterrupt\")\n    assert conventional(\"praise\", \"thanks for the test!\") == \"praise: thanks for the test!\"\n    for args, message in [((\"rant\", \"x\"), \"unknown label: rant\"), ((\"issue\", \"x\", \"urgent\"), \"unknown decoration: urgent\")]:\n        try:\n            conventional(*args)\n        except ValueError as e:\n            assert str(e) == message\n        else:\n            raise AssertionError(message)")

### Exercise: Risky patterns

Using the given `added_lines`, write `risky_changes(diff)`: for each added line outside `Lib/test/`, report `"PATH:LINE: MESSAGE"` for each rule it matches, in this order: a bare `except:` (`"a bare except: catches everything, including KeyboardInterrupt"`), `== None` or `!= None` (`"compare with None using is or is not"`), a `def` with a default of `[]`, `{}` or `set()` (`"a mutable default argument is shared between calls"`), `print(` (`"a print left in library code"`), and `eval(` (`"eval on data can run any code"`).

In [ ]:
import re
HUNK = re.compile(r"@@ -\d+(?:,\d+)? \+(\d+)(?:,\d+)? @@")

def added_lines(diff):
    path, number = None, 0
    for line in diff.splitlines():
        if line.startswith("diff --git "):
            path = line.split(" b/", 1)[1]
        elif line.startswith(("+++", "---")):
            continue
        elif m := HUNK.match(line):
            number = int(m.group(1))
        elif line.startswith("+"):
            yield path, number, line[1:]
            number += 1
        elif line.startswith(" "):
            number += 1

def risky_changes(diff):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_pr():\n    \"four problems in the new function\"\n    assert risky_changes(open(\"pr.diff\").read()) == [\n        \"Lib/wraptext.py:29: a mutable default argument is shared between calls\",\n        \"Lib/wraptext.py:34: a bare except: catches everything, including KeyboardInterrupt\",\n        \"Lib/wraptext.py:36: compare with None using is or is not\",\n        \"Lib/wraptext.py:37: a print left in library code\"]\n\ndef test_more():\n    \"other shapes\"\n    diff = \"diff --git a/Lib/x.py b/Lib/x.py\\n--- a/Lib/x.py\\n+++ b/Lib/x.py\\n@@ -1 +1,3 @@\\n+def f(a, opts={}):\\n+    return eval(a) if a != None else 0\\n+except Exception:\\n\"\n    assert risky_changes(diff) == [\"Lib/x.py:1: a mutable default argument is shared between calls\",\n                                   \"Lib/x.py:2: compare with None using is or is not\", \"Lib/x.py:2: eval on data can run any code\"]")

### Exercise: New functions without docstrings

Using the given `added_lines`, write `undocumented_functions(diff)`: for each added line outside `Lib/test/` that defines a public function (`def NAME(` with NAME not starting with `_`), report `"PATH:LINE: NAME has no docstring"` unless the next **added** line in the same file, stripped, starts with three quote marks (`"""` or `'''`).

In [ ]:
import re
HUNK = re.compile(r"@@ -\d+(?:,\d+)? \+(\d+)(?:,\d+)? @@")

def added_lines(diff):
    path, number = None, 0
    for line in diff.splitlines():
        if line.startswith("diff --git "):
            path = line.split(" b/", 1)[1]
        elif line.startswith(("+++", "---")):
            continue
        elif m := HUNK.match(line):
            number = int(m.group(1))
        elif line.startswith("+"):
            yield path, number, line[1:]
            number += 1
        elif line.startswith(" "):
            number += 1

def undocumented_functions(diff):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_pr():\n    \"a public function needs a docstring\"\n    assert undocumented_functions(open(\"pr.diff\").read()) == [\"Lib/wraptext.py:29: shorten_all has no docstring\"]\n    diff = 'diff --git a/Lib/x.py b/Lib/x.py\\n--- a/Lib/x.py\\n+++ b/Lib/x.py\\n@@ -1 +1,4 @@\\n+def f():\\n+    \"\"\"Do it.\"\"\"\\n+def _g():\\n+    pass\\n'\n    assert undocumented_functions(diff) == []")

### Exercise: Kind and specific

Write `tone_problems(comment)`, ignoring case: report `"about the person: PHRASE"` for each of `you forgot`, `you didn't`, `why didn't you` and `you should have` found; `"dismissive: WORD"` for each of `just`, `obviously`, `simply` and `clearly` as whole words; and `"no label"` unless the comment starts with one of the Conventional Comments labels followed by `:` or ` (`.

In [ ]:
import re

def tone_problems(comment):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_tone():\n    \"describe the code, not the person\"\n    assert tone_problems(\"issue (blocking): this except: also catches KeyboardInterrupt; what about except ValueError?\") == []\n    assert tone_problems(\"You forgot the test. Just add one, obviously.\") == [\n        \"about the person: you forgot\", \"dismissive: just\", \"dismissive: obviously\", \"no label\"]\n    assert tone_problems(\"question: why didn't you use is None?\") == [\"about the person: why didn't you\"]")

### Exercise: Summing up

Write `review_summary(comments)`: each comment is a Conventional Comments string, perhaps with a `PATH:LINE: ` prefix. Count comments by label, in the order `issue`, `suggestion`, `question`, `nitpick`, `thought`, `praise` (leaving out labels with none), and count the blocking ones. Return `(counts, verdict)`: `"request changes"` if any comment is blocking, `"comment"` if there are any issues, suggestions or questions, otherwise `"approve"`.

In [ ]:
import re

def review_summary(comments):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_summary():\n    \"a clear verdict\"\n    comments = [\"Lib/wraptext.py:34: issue (blocking): a bare except\", \"Lib/wraptext.py:29: issue (blocking): a shared default\",\n                \"suggestion (non-blocking): add a docstring\", \"nitpick: blank lines\", \"praise: thanks for the test!\"]\n    assert review_summary(comments) == ({\"issue\": 2, \"suggestion\": 1, \"nitpick\": 1, \"praise\": 1, \"blocking\": 2}, \"request changes\")\n    assert review_summary([\"question: is None skipped on purpose?\", \"praise: neat\"])[1] == \"comment\"\n    assert review_summary([\"nitpick: naming\", \"praise: neat\"]) == ({\"nitpick\": 1, \"praise\": 1, \"blocking\": 0}, \"approve\")")

### Bug hunt: A shared default that slipped by

The check only looks for `=[]` without a space, so `seen = []` gets through and this prints `False`. Allow spaces around `=` so it prints `True`.

In [ ]:
import re

line = "def shorten_all(texts, width, seen = []):"
print(bool(re.search(r"def \w+\(.*=\[\]", line)))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"\\\\s*\"\n    expect_output(\"True\")")

### Bug hunt: Just, with a capital

The comment starts with `Just`, but the check is case-sensitive, so it prints `[]`. Ignore case so it prints `['just']`.

In [ ]:
import re

comment = "Just use is None here."
print([w for w in ["just", "obviously"] if re.search(rf"\b{w}\b", comment)])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"ignore case\"\n    expect_output(\"['just']\")")

### Puzzle: Choosing the verdict

Arrange the end of `review_summary`. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # what shorten_all(["a"]), shorten_all(["b"]) print in the warm-up, as a list of two ints
q = ___   # the right way to write `x == None`
r = ___   # the verdict for a review with one blocking issue
s = ___   # True or False: you must be a maintainer to review a pull request
t = ___   # the Conventional Comments label for an optional point of style

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == [1, 2] and q == \"x is None\" and r == \"request changes\", \"p, q and r\"\n    assert s is False and t == \"nitpick\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: reviewing the pull request

Write `review_pr(diff)` with the given helpers. Collect comments in this order: an `issue (blocking)` for each risky change (with its `PATH:LINE` as `where`); a `suggestion (non-blocking)` for each undocumented function, subject `"add a docstring to NAME"` with its `where`; then, looking at the changed files (paths from `diff --git` lines), an `issue (blocking)` `"a new function needs a NEWS entry"` if no path starts with `Misc/NEWS.d/` and some added `Lib/` (not test) line defines a public function; `suggestion (non-blocking)` `"document the new function in Doc/library/"` in the same case if no path starts with `Doc/`; and `praise` `"thanks for adding a test"` if a path starts with `Lib/test/`. Return `(comments, verdict)`, the verdict from `review_summary`.

In [ ]:
import re

import re
HUNK = re.compile(r"@@ -\d+(?:,\d+)? \+(\d+)(?:,\d+)? @@")

def added_lines(diff):
    path, number = None, 0
    for line in diff.splitlines():
        if line.startswith("diff --git "):
            path = line.split(" b/", 1)[1]
        elif line.startswith(("+++", "---")):
            continue
        elif m := HUNK.match(line):
            number = int(m.group(1))
        elif line.startswith("+"):
            yield path, number, line[1:]
            number += 1
        elif line.startswith(" "):
            number += 1

RULES = [
    (r"^\s*except\s*:", "a bare except: catches everything, including KeyboardInterrupt"),
    (r"[=!]= None\b", "compare with None using is or is not"),
    (r"def \w+\(.*=\s*(\[\]|\{\}|set\(\))", "a mutable default argument is shared between calls"),
    (r"\bprint\(", "a print left in library code"),
    (r"\beval\(", "eval on data can run any code"),
]

def risky_changes(diff):
    found = []
    for path, n, text in added_lines(diff):
        if path.startswith("Lib/test/"):
            continue
        for pattern, message in RULES:
            if re.search(pattern, text):
                found.append(f"{path}:{n}: {message}")
    return found

def conventional(label, subject, decoration=None, where=None):
    text = f"{label} ({decoration}): {subject}" if decoration else f"{label}: {subject}"
    return f"{where}: {text}" if where else text

def review_summary(comments):
    labels = [m.group(1) for c in comments if (m := re.search(r"(?:^|: )(praise|nitpick|suggestion|issue|question|thought)(?: \(([\w-]+)\))?: ", c))]
    blocking = sum("(blocking)" in c for c in comments)
    if blocking:
        return "request changes"
    return "comment" if any(l in labels for l in ("issue", "suggestion", "question")) else "approve"

def review_pr(diff):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_pr():\n    \"a kind, clear review\"\n    comments, verdict = review_pr(open(\"pr.diff\").read())\n    assert comments == [\n        \"Lib/wraptext.py:29: issue (blocking): a mutable default argument is shared between calls\",\n        \"Lib/wraptext.py:34: issue (blocking): a bare except: catches everything, including KeyboardInterrupt\",\n        \"Lib/wraptext.py:36: issue (blocking): compare with None using is or is not\",\n        \"Lib/wraptext.py:37: issue (blocking): a print left in library code\",\n        \"Lib/wraptext.py:29: suggestion (non-blocking): add a docstring to shorten_all\",\n        \"issue (blocking): a new function needs a NEWS entry\",\n        \"suggestion (non-blocking): document the new function in Doc/library/\",\n        \"praise: thanks for adding a test\",\n    ]\n    assert verdict == \"request changes\"\n\ndef test_clean():\n    \"a docs-only change\"\n    diff = \"diff --git a/Doc/library/wraptext.rst b/Doc/library/wraptext.rst\\n--- a/Doc/library/wraptext.rst\\n+++ b/Doc/library/wraptext.rst\\n@@ -1 +1 @@\\n-Shorten text.\\n+Shorten text to fit a width.\\n\"\n    assert review_pr(diff) == ([], \"approve\")")

## Warm-up

The default list is created **once**, when the function is defined, and shared by every call, so `seen` keeps growing. `pr.diff` adds a function with exactly that problem, among others. Reading it as a reviewer, you'll find them before they're merged.

## Why review

Review is how projects keep their quality while accepting changes from strangers, and it's the bottleneck in most of them. Anyone can review: CPython welcomes reviews from contributors, and a pull request with a helpful review from someone else is easier for a core developer to finish. You learn the codebase fast, too: every review is a guided tour of a change. Be clear that you aren't a maintainer, and leave the final decision to them.

## What to look for

Read the issue first, then the description, then the diff. Ask, in order:

1. **Should this change happen at all?** Does it solve the issue, and is it the right approach?
2. **Is it correct?** Run it. Try edge cases. Look for risky patterns: a bare `except:` (which catches `KeyboardInterrupt` too), `== None` (use `is None`), a **mutable default argument**, a leftover `print`, `eval` on data.
3. **Is it tested?** A regression test for a fix; tests for a feature.
4. **Is it complete?** Docstring and documentation for a new public function, a NEWS entry, a version note in the docs.
5. **Is it clear?** Names, structure, comments. These matter least: say so.

## Writing comments

Say what kind of comment each one is and whether it blocks. **Conventional Comments** (conventionalcomments.org, by Paul Slaughter) gives a simple format: a label, an optional decoration, and the subject, such as `issue (blocking): a bare except: hides KeyboardInterrupt`, `suggestion (non-blocking): add a docstring`, `nitpick: two blank lines here`, `question: should None be skipped or kept?`, `praise: thanks for the test!`. The author then knows exactly what they must do and what they may ignore.

Kind and specific beats short and sharp. Describe the code, not the person ("this branch never runs", not "you forgot"); suggest rather than order ("what about `is None`?"); avoid "just", "obviously" and "simply"; and say what's good, sincerely.

## Summing up

End with a summary: **approve** if nothing blocks, **comment** if there are suggestions or questions only, **request changes** if anything blocks. One sentence on the overall shape helps: "The idea is good and the test is a great start; the bare `except:` and the shared default need fixing before merge."

## Practice

## Quiz

## Daily challenge

## Your turn

1. **Find an open pull request** in a project you know (CPython's `docs` or `tests` pull requests are a good start) that has no review yet.
2. **Run it**: check out the branch (`gh pr checkout NNN` with GitHub's free command-line tool, or fetch it with Git) and try its edge cases.
3. **Review it** in the five steps, with labelled comments, at least one sincere `praise`, and a summary. Say you're a contributor, not a maintainer.
4. **Log the review** for Day 5's write-up.

## Before you go

Tomorrow you'll write up what you've contributed: a log, the numbers, and a post that others (and future you) can learn from.